# 03 · Gate-Diagnose & neue Angriffspunkte

**Bachelorarbeit: AF-Detektion in kontaktlosen Signalen · Nik Büttner · RWTH Aachen**

Ausgangslage aus `02_experts_gating2`:
- **Experten sind komplementär** (gut!): PPG-allein stacked 0.826 → +cECG 0.875 → +BCG 0.889 → alle 0.897.
  Multimodal lohnt sich also; das Problem ist **isoliert das Gewichtungs-Gate**.
- **Gate schlägt equal nicht** (equal 0.939, gb-win 0.937, gb-fix 0.940). torch_mlp 0.944 ist der einzige knappe Vorsprung.
- **Ursache:** SQI sagt die Zuverlässigkeit kaum vorher — v.a. BCG (r≈0.05–0.07 für cosen/drr).
- **BCG-Detektion** ist der Flaschenhals: hr_err ~6 bpm auch bei guter GT (cECG ~2, PPG ~1.5).
- **GT selbst ist verrauscht**: nur 31 % der Fenster mit gt_conf≥0.80, Median 0.71.

Dieses Notebook setzt drei konkrete, literatur­gestützte Angriffe um — **alle lauffähig auf
den vorhandenen Caches**, ohne `src/`-Änderung:
1. **Welche SQIs helfen wirklich?** (Feature-Audit + Pruning-Test) — *Gemini-Hinweis*
2. **Welche 3–5 Patienten sind „Schrott"?** (Patienten-Ranking + Ausschluss-Re-Eval)
3. **Gate als KLASSIFIKATION** (zuverlässig/unzuverlässig) statt Fehler-Regression — *aus der Literatur*

Plus ein BCG-Experiment (detektionsfreie Features statt J-Zacken-RR) und die Literatur-Notizen.

## 1 · Setup — Caches laden (Feature-, Reliability-, GT-Güte-Tabelle, Fold-Cache)

In [11]:
import sys, os

# Set working directory to your project root
os.chdir('/home/nik/projects/BA')

# Add both root and src to path
sys.path.insert(0, '/home/nik/projects/BA')
sys.path.insert(0, '/home/nik/projects/BA/src')

In [12]:
import os, sys, glob
import numpy as np, pandas as pd
SRC_DIR = os.path.abspath(os.path.join(os.path.abspath(''), '..', 'src'))
if SRC_DIR not in sys.path: sys.path.insert(0, SRC_DIR)
import extract as E, experts as X, reliability as R, gating as G, fusion_cv as CV, models as M
from sklearn.metrics import roc_auc_score

DATA_DIR='results/'
df = pd.read_csv(sorted(glob.glob(os.path.join(DATA_DIR,'features_sqi_cecg_cwt_24fab5.csv')), key=os.path.getmtime)[-1])
df, y, groups = E.split_Xygroups(df)
rel = pd.read_csv(os.path.join(DATA_DIR,'reliability_cosen.csv'))
gtq = pd.read_csv(os.path.join(DATA_DIR,'gt_quality.csv'))
rel_al = CV.align_reliability(df, rel)
gate_cols = E.gate_sqi_cols(df, 'all')
fold_cache = CV.precompute_folds(df, y, groups, inner_splits=5, random_state=42, n_jobs=-1, cache_dir=DATA_DIR)
print('df', df.shape, '· Gate-Eingänge', len(gate_cols), '· Folds', len(fold_cache['folds']))

Fold-Cache gefunden -> lade results/fold_expert_cache_399e0d3253908131.joblib
df (4724, 205) · Gate-Eingänge 30 · Folds 40


## 2 · Welche SQIs helfen — und welche sind Rauschen? (Gemini-Hinweis)

Pro Gate-Eingang die **Rang-Korrelation (Spearman)** mit dem Zuverlässigkeits-ZIEL je
Modalität, nur auf GT-gültigen Fenstern. Klein über alle Modalitäten/Ziele ⇒ das Feature
trägt nichts zur Gewichtung bei und verdünnt nur den Gate-Eingang. Anschließend Test:
hebt das **Entfernen** dieser Rausch-Eingänge das Gate-r?

In [13]:
from scipy.stats import spearmanr
def audit(target):
    rows=[]
    for c in gate_cols:
        d={'feature':c}
        for m in G.ORDER:
            ok=(rel_al[f'rel_{m}_valid']==True) & np.isfinite(rel_al[f'rel_{m}_{target}']) & np.isfinite(df[c])
            d[f'rho_{m}']=spearmanr(df.loc[ok,c], rel_al.loc[ok,f'rel_{m}_{target}']).correlation if ok.sum()>50 else np.nan
        rows.append(d)
    a=pd.DataFrame(rows).set_index('feature')
    a['max|rho|']=a.abs().max(axis=1)
    return a.sort_values('max|rho|', ascending=False)

aud = audit('hr_err')   # hr_err = das vom SQI am besten vorhergesagte Ziel (deine Tabelle)
print('Top-12 informativste Gate-Eingänge (Ziel hr_err):')
print(aud.head(12).round(3).to_string())
noise = aud.index[aud['max|rho|'] < 0.05].tolist()
print(f'\n{len(noise)} Rausch-Kandidaten (|rho|<0.05 für ALLE Modalitäten):\n', noise)

Top-12 informativste Gate-Eingänge (Ziel hr_err):
                    rho_cecg  rho_ppg  rho_bcg  max|rho|
feature                                                 
sqi_ppg1_tSQI         -0.437   -0.533    0.006     0.533
sqi_ppg1_kSQI          0.236    0.525    0.385     0.525
sqi_ppg1_pSQI         -0.262   -0.479   -0.291     0.479
sqi_ppg2_kSQI          0.230    0.477    0.380     0.477
sqi_ppg2_pSQI         -0.239   -0.438   -0.308     0.438
sqi_ppg_xcorr         -0.264   -0.424   -0.268     0.424
sqi_cecg_bSQI         -0.355   -0.228   -0.119     0.355
sqi_ppg2_tSQI         -0.309   -0.340    0.106     0.340
sqi_ppg2_bSQI         -0.082    0.048    0.296     0.296
sqi_ppg1_bSQI         -0.192   -0.076    0.292     0.292
sqi_cecg_kSQI         -0.049    0.263    0.219     0.263
sqi_xmodal_n_agree    -0.005   -0.140   -0.262     0.262

1 Rausch-Kandidaten (|rho|<0.05 für ALLE Modalitäten):
 ['sqi_bcg1_sSQI']


In [14]:
# Pruning-Test: Gate-r MIT vs. OHNE die Rausch-Eingänge (gleiche Routine wie im Einsatz).
for tag, dd in [('alle SQIs', df), ('ohne Rausch-SQIs', df.drop(columns=noise))]:
    pv = CV.gate_predictive_validity(dd, rel, y, groups, gate_kind='gb', target_metric='drr_sd_err')
    print(f'{tag:18s} r = {pv["r"].round(3).to_dict()}')
print('\nLesart: steigt r (v.a. cECG/PPG) ohne die Rausch-Eingänge, fliegen sie dauerhaft raus '
      '(gate_sqi_cols entsprechend einschränken oder Spalten vor evaluate_moe droppen).')

alle SQIs          r = {'cecg': 0.457, 'ppg': 0.287, 'bcg': 0.054}
ohne Rausch-SQIs   r = {'cecg': 0.456, 'ppg': 0.265, 'bcg': 0.077}

Lesart: steigt r (v.a. cECG/PPG) ohne die Rausch-Eingänge, fliegen sie dauerhaft raus (gate_sqi_cols entsprechend einschränken oder Spalten vor evaluate_moe droppen).


## 3 · „Schrott-Patienten" finden und ausschließen

Vermutung: 3–5 Patienten mit durchgehend schlechtem Signal ziehen alles runter. Wir ranken
Patienten nach einem transparenten Badness-Score aus drei GT-/qualitätsbezogenen Größen:
mittlerer Experten-Fehler (alle Modalitäten), niedrige GT-Konfidenz, niedrige beste ACF-Peak-
Höhe (Detektierbarkeit). Dann: equal-Fusion-AUC mit/ohne die schlechtesten k.

In [15]:
oof = X.oof_expert_probs(df, y, groups, n_splits=5)
per = pd.DataFrame({'patient':groups, 'win_idx':df['win_idx'].values, 'AF':y})
for m in G.ORDER: per[f'err_{m}'] = np.abs(y - oof[f'p_{m}'].values)
SIGS={'cecg':['cecg'],'ppg':['ppg1','ppg2'],'bcg':['bcg1','bcg2']}
for m,sigs in SIGS.items():
    per[f'acf_{m}'] = df[[f'{s}_acf_peak' for s in sigs]].max(axis=1).values
per = per.merge(gtq[['patient','win_idx','gt_conf']], on=['patient','win_idx'], how='left')

g = per.groupby('patient')
rep = pd.DataFrame({
    'AF':            g['AF'].first(),
    'err_mean':      g[[f'err_{m}' for m in G.ORDER]].mean().mean(axis=1),
    'gt_conf':       g['gt_conf'].mean(),
    'acf_best':      g[[f'acf_{m}' for m in G.ORDER]].mean().max(axis=1),
})
# z-standardisierter Badness-Score: hoher Fehler + niedrige GT + niedrige ACF
z=lambda s:(s-s.mean())/s.std()
rep['badness'] = z(rep['err_mean']) + z(1-rep['gt_conf']) + z(1-rep['acf_best'])
rep = rep.sort_values('badness', ascending=False)
print('Schlechteste 8 Patienten:'); print(rep.head(8).round(3).to_string())

Schlechteste 8 Patienten:
         AF  err_mean  gt_conf  acf_best  badness
patient                                          
PAT042    0     0.754    0.479     0.437    5.470
PAT038    1     0.422    0.419     0.682    2.800
PAT036    1     0.525    0.746     0.327    2.716
PAT026    1     0.206    0.609     0.320    1.898
PAT006    0     0.444    0.680     0.541    1.696
PAT030    1     0.190    0.657     0.279    1.649
PAT043    0     0.300    0.450     0.757    1.499
PAT010    0     0.520    0.840     0.433    1.487


In [16]:
# Wirkung des Ausschlusses auf die equal-Fusion (window-AUC, threshold-frei).
fused = oof[[f'p_{m}' for m in G.ORDER]].mean(axis=1).values
print('equal-Fusion window-AUC bei Ausschluss der schlechtesten k Patienten:')
for k in [0,2,3,4,5]:
    drop=set(rep.index[:k]); keep=~pd.Series(groups).isin(drop).values
    print(f'  k={k}  ({", ".join(sorted(drop)) or "—"}):  AUC={roc_auc_score(y[keep], fused[keep]):.3f}  (n={keep.sum()})')
print('\nHinweis: das ist die SELEKTIONS-Wirkung (gleiche Experten, nur Test-Set verkleinert). '
      'Für die DEPLOYMENT-Wirkung (Experten ohne die Patienten neu trainiert) unten compare_gates '
      'mit dem reduzierten df/rel/y/groups und fold_cache=None laufen lassen.')

equal-Fusion window-AUC bei Ausschluss der schlechtesten k Patienten:
  k=0  (—):  AUC=0.918  (n=4724)
  k=2  (PAT038, PAT042):  AUC=0.943  (n=4486)
  k=3  (PAT036, PAT038, PAT042):  AUC=0.950  (n=4367)
  k=4  (PAT026, PAT036, PAT038, PAT042):  AUC=0.948  (n=4248)
  k=5  (PAT006, PAT026, PAT036, PAT038, PAT042):  AUC=0.952  (n=4129)

Hinweis: das ist die SELEKTIONS-Wirkung (gleiche Experten, nur Test-Set verkleinert). Für die DEPLOYMENT-Wirkung (Experten ohne die Patienten neu trainiert) unten compare_gates mit dem reduzierten df/rel/y/groups und fold_cache=None laufen lassen.


In [17]:
# Optional (teurer, ~80s): Experten OHNE die schlechtesten k neu trainieren und Fusion messen.
k = 4
drop = set(rep.index[:k]); keep = ~pd.Series(groups).isin(drop).values
df_k = df[keep].reset_index(drop=True)
y_k, g_k = y[keep], groups[keep]
tab_k = CV.compare_gates(df_k, rel, y_k, g_k, gate_kinds=('equal','gb'),
                         target_metric='cosen_err', inner_splits=5, min_spec=0.80)  # fold_cache=None -> neu
print(f'Ohne {sorted(drop)}:')
print(tab_k[['gate','AUC','Sensitivität','Spezifität','Accuracy']].round(3).to_string(index=False))

Ohne ['PAT026', 'PAT036', 'PAT038', 'PAT042']:
 gate   AUC  Sensitivität  Spezifität  Accuracy
equal 0.955         0.951       0.798     0.867
   gb 0.950         0.958       0.789     0.864


## 4 · Gate als KLASSIFIKATION (zuverlässig / unzuverlässig)

Aus der Literatur (Naeini SQUWA 2024; Pereira/ACM 2023; *Assessing impact of signal quality
on HR under varying rhythms* 2025): Signalqualität wird viel zuverlässiger als **binäre
Klassifikation** (gutes/schlechtes Fenster) gelernt als per Fehler-**Regression** — dort sind
AUROCs von 0.96 berichtet, während unsere Fehler-Regression bei r≈0.05–0.46 hängt.

Reframing: pro Modalität & Fenster Label „zuverlässig" = (GT-gültig **und** Fehler ≤
Trainings-Median). Ein Klassifikator sagt P(zuverlässig) aus dem SQI vorher; das wird direkt
zum Fusionsgewicht (normiert). Leckagefrei über den vorhandenen `fold_cache` (LOPO-Experten).

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

def clf_gate_fusion(target='hr_err', clf='lr'):
    yt, yp_clf, yp_eq = [], [], []
    for f in fold_cache['folds']:
        tr, te, P_tr, P_te = f['tr'], f['te'], f['P_tr'], f['P_te']
        Xtr, Xte = df.iloc[tr][gate_cols].values, df.iloc[te][gate_cols].values
        W = np.ones((len(te), len(G.ORDER)))
        for j, m in enumerate(G.ORDER):
            v = (rel_al.iloc[tr][f'rel_{m}_valid']==True).to_numpy()
            e = rel_al.iloc[tr][f'rel_{m}_{target}'].values.astype(float)
            good = v & np.isfinite(e)
            if good.sum() < 20:
                continue
            thr = np.median(e[good])
            lab = (good & (e <= thr)).astype(int)
            if lab.sum() in (0, len(lab)):
                continue
            pipe = (Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                              ('c',LogisticRegression(max_iter=1000,class_weight='balanced'))])
                    if clf=='lr' else
                    Pipeline([('imp',SimpleImputer(strategy='median')),
                              ('c',HistGradientBoostingClassifier(random_state=42))]))
            pipe.fit(Xtr, lab)
            W[:, j] = np.clip(pipe.predict_proba(Xte)[:, 1], 1e-3, None)
        W = W / W.sum(axis=1, keepdims=True)
        yt.extend(y[te]); yp_clf.extend((W*P_te).sum(axis=1)); yp_eq.extend(P_te.mean(axis=1))
    return roc_auc_score(yt, yp_clf), roc_auc_score(yt, yp_eq)

for tgt in ['hr_err','drr_sd_err','cosen_err']:
    for c in ['lr','gb', 'mlp']:
        a_clf, a_eq = clf_gate_fusion(tgt, c)
        print(f'  Klassifikations-Gate  Ziel={tgt:10s} clf={c:3s}: AUC={a_clf:.3f}   (equal={a_eq:.3f})')
print('\nLesart: schlägt eine Variante equal (0.939) deutlich, ist das Klassifikations-Gate der '
      'bessere Weg als die Fehler-Regression — dann sauber in gating.py/fusion_cv.py gießen.')

  Klassifikations-Gate  Ziel=hr_err     clf=lr : AUC=0.925   (equal=0.939)
  Klassifikations-Gate  Ziel=hr_err     clf=gb : AUC=0.923   (equal=0.939)


  Klassifikations-Gate  Ziel=hr_err     clf=mlp: AUC=0.923   (equal=0.939)
  Klassifikations-Gate  Ziel=drr_sd_err clf=lr : AUC=0.923   (equal=0.939)
  Klassifikations-Gate  Ziel=drr_sd_err clf=gb : AUC=0.939   (equal=0.939)
  Klassifikations-Gate  Ziel=drr_sd_err clf=mlp: AUC=0.939   (equal=0.939)
  Klassifikations-Gate  Ziel=cosen_err  clf=lr : AUC=0.936   (equal=0.939)
  Klassifikations-Gate  Ziel=cosen_err  clf=gb : AUC=0.931   (equal=0.939)
  Klassifikations-Gate  Ziel=cosen_err  clf=mlp: AUC=0.931   (equal=0.939)

Lesart: schlägt eine Variante equal (0.939) deutlich, ist das Klassifikations-Gate der bessere Weg als die Fehler-Regression — dann sauber in gating.py/fusion_cv.py gießen.


## 5 · BCG-Experte OHNE J-Zacken-RR (detektionsfrei)

Die BCG-AF-Literatur (Jung 2025 IOPscience, 116 Pat., AUROC 0.97; Multi-Site-BCG 2025; FabosNet
2023) **vermeidet die J-Zacken-Detektion** und nutzt Autokorrelation / Leistungsspektrum —
genau weil J-Zacken bei AF „unobtainable" sind (deckt sich mit deinem hr_err≈6 bpm). Test:
BCG-Experte nur mit **detektionsfreien** Features (Zeit/Frequenz/Spektral-Entropie/ACF), ohne
RR-/HRV-Features. Schlägt das den RR-basierten BCG-Experten (OOF 0.809)?

In [19]:
from sklearn.model_selection import StratifiedGroupKFold
RR_SUFFIX = ('meanRR','SDNN','RMSSD','pNN50','SD1','SD2','SD1_SD2','CoSEn','rrShannon',
             'TPR','DFA_a1','Lorenz_occ','Lorenz_origin','dRR_SD','RR_CV','RMSSDn','dRR_MAD')
def oof_auc_cols(cols):
    Xc=df[cols].values; oof=np.full(len(df), np.nan)
    skf=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    for tr,va in skf.split(Xc,y,groups):
        pipe=Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                       ('c',LogisticRegression(max_iter=1000,class_weight='balanced'))])
        pipe.fit(Xc[tr],y[tr]); oof[va]=pipe.predict_proba(Xc[va])[:,1]
    return roc_auc_score(y,oof)

bcg_all = E.expert_feature_cols(df,'bcg')
bcg_nodet = [c for c in bcg_all if not c.endswith(RR_SUFFIX)]
print(f'BCG alle Features         ({len(bcg_all):2d}): OOF-AUC {oof_auc_cols(bcg_all):.3f}')
print(f'BCG detektionsfrei        ({len(bcg_nodet):2d}): OOF-AUC {oof_auc_cols(bcg_nodet):.3f}')
print('\nNächster Schritt bei Erfolg: BCG-Spektralfeatures ausbauen (Leistung in 0.7–10 Hz-Bändern, '
      'spektrale Konzentration) und volle ACF-Kurve (Nebengipfel-Verhältnis) — beide ohne Peak-Detektion.')

BCG alle Features         (66): OOF-AUC 0.801
BCG detektionsfrei        (32): OOF-AUC 0.792

Nächster Schritt bei Erfolg: BCG-Spektralfeatures ausbauen (Leistung in 0.7–10 Hz-Bändern, spektrale Konzentration) und volle ACF-Kurve (Nebengipfel-Verhältnis) — beide ohne Peak-Detektion.


## 6 · Literatur-Angriffspunkte (für Schreibteil & nächste Sitzungen)

**A — Gate als Klassifikation statt Regression** *(Abschnitt 4)*
- *SQUWA* (Naeini et al., arXiv:2404.15353, 2024): Qualität als **Attention-Gewicht im
  AF-Detektor selbst**, nicht als getrennter Vorschritt — „moving away from the detached,
  two-step methods". Genau unser Problem.
- *DL PPG Quality Assessment* (Pereira et al., ACM THC 2023): binäre Reliable/Unreliable-Labels
  je Feature via Schwellen-Vergleich zum EKG — Vorbild fürs Label-Design.
- *Assessing impact of signal quality on HR under varying rhythms* (Biomed. Signal Process.
  Control 2025): handcrafted features + **Feature-Selektion** + 7 Klassifikatoren, AUROC 0.96;
  Autokorrelation für AF-Rhythmus.

**B — BCG ohne J-Zacken** *(Abschnitt 5)*
- *Bed-sensor BCG für AF* (Jung et al., Physiol. Meas. 2025, IOPscience): **Autokorrelation**
  statt J-Zacken, AUROC 0.97 (116 Pat.). „peaks can be unobtainable when signal is non-periodic".
- *Multi-Site BCG AF* (Sensors 2025, 25(22):6833): **Leistungsspektrum <10 Hz** → ML, Acc 0.92.
- *FabosNet* (Biomed. Signal Process. Control 2023): self-supervised BCG-HRV, AF-Trennung 5–30 s.

**C — Gelernte Cross-Modal-Fusion** (größerer Umbau, starke Thesis-Richtung)
- *MM-TFNet* (Sensors 2026, 26(2):548): **dieselbe Art 40-Personen-BCG/PPG/EKG-Datenbank**,
  Cross-Modal-Attention (TCN+BiLSTM+MHSA), HR-MAE 0.88 bpm. Lernt die Modalitätsgewichte
  end-to-end statt über einen SQI-Fehler-Umweg.

---

### Warum Bachelet r=0.89 (HR) und wir nicht
1. **Anderes Ziel.** Sein Ziel war der HR-Fehler — eine glatte, gut vorhersagbare Größe. Unser
   AF-Ziel (cosen_err/drr_sd_err = RR-Irregularitäts-Fehler) ist intrinsisch verrauschter.
   **Deine eigene Tabelle bestätigt das:** hr_err ist auch bei uns am besten vorhersagbar
   (cECG 0.34, PPG 0.42), cosen_err am schlechtesten (cECG 0.04). Wir kämpfen gegen das Ziel.
2. **Aber selbst auf HR sind wir bei ~0.42, nicht 0.89** → der Rest ist **Detektion/Features**:
   Reliabilitäts-Prädiktion erreicht in der Literatur mit EINEM SQI nur AUC ~0.75
   (Sliding-Scale-SQI 2023). r=0.89 ist außergewöhnlich und deutet auf eine **deutlich bessere
   Detektion** (sein HR-Fehler war strukturierter) und/oder **HR-spezifisch getunte Features**
   und/oder eine **gepoolte (nicht-LOPO) Auswertung**.

**Fragen an Nico:** (1) War Bachelets r=0.89 LOPO oder gepoolt? (2) Welche SQI-Features genau?
(3) Ist ein **Klassifikations-Gate** (zuverlässig/unzuverlässig) als Abweichung von Bachelet
akzeptabel? (4) Dürfen 3–5 Patienten mit nachweislich schlechtem Signal ausgeschlossen werden?

In [20]:
CSV_30S = os.path.join(DATA_DIR, 'features_sqi_cecg_cwt_24fab5.csv')  # 30s/15s, 4724 Fenster
df = pd.read_csv(CSV_30S); df, y, groups = E.split_Xygroups(df)
assert len(df) < 6000, f'Falsche Tabelle (n={len(df)}) — das ist die 10-s-Tabelle!'